# Train RNN decoders

Trains the RNN movement decoder serially on a single GPU. Two modes selected via the `DEMO_MODE` toggle in the next cell:

- **Demo mode** (`DEMO_MODE = True`) — trains **5 folds × 3 seed = 15 models** in series. Demonstrates the pipeline end-to-end without committing large compute resources. Won't reproduce the paper figure exactly (the 10-seed averaging is part of the published methodology).
- **Full mode** (`DEMO_MODE = False`) — trains **5 folds × 10 seeds = 50 models**. Required to more closely reproduce Figure 3 from the paper.

The training loop skips any (fold, seed) whose `done` marker exists, so re-running picks up where it left off after an interruption. Each run writes its outputs to `outputs/training/fold_<fold>_4sec/seed=<seed>/`:

- `modelWeights` — best-test-accuracy state dict (PyTorch `state_dict`)
- `trainingStats` — per-eval-step loss / accuracy history (pickled)
- `done` — completion marker; presence means the full `nBatch` loop finished

Notebook 3 reads from these paths and adapts automatically to whatever number of seeds you trained.

In [ ]:
# ============================================================
# Toggle here
# ============================================================
DEMO_MODE = True       # True = 5 folds × 3 seed (15 models); False = 5 folds × 10 seeds (50 models)
# ============================================================

from whole_body_pipeline import OUTPUT_DIR, N_FOLDS

SEEDS_LIST = [0,1,2] if DEMO_MODE else list(range(10))

formatted_data_dir = OUTPUT_DIR / 'formatted_data'
training_dir       = OUTPUT_DIR / 'training'

print(f'DEMO_MODE = {DEMO_MODE}')
print(f'Seeds per fold: {SEEDS_LIST}')
print(f'Folds: {N_FOLDS}; total models to train: {N_FOLDS * len(SEEDS_LIST)}')
print(f'Formatted data:  {formatted_data_dir}')
print(f'Training output: {training_dir}')

## Train

Iterates over every `(fold, seed)` combination in series, calling `trainModel` once per combination. The loop:

- **Reads the base config** from `conf/config.yaml` and overrides `datasetPath`, `outputDir`, and `seed` for each run.
- **Skips runs that have already completed** by checking for the `done` marker file in each `seed=*/` folder. Safe to re-run after interruption.
- **Runs in this Python process**, so closing the notebook (or losing the kernel) stops training. Use `tmux`/`screen` if your environment is fragile.

In [ ]:
from pathlib import Path
from copy import deepcopy

import yaml

from neural_decoder_trainer import trainModel

with open(Path('conf') / 'config.yaml') as handle:
    base_cfg = yaml.safe_load(handle)
base_cfg.pop('hydra', None)               # only used by the Hydra entry point

n_total = N_FOLDS * len(SEEDS_LIST)
completed_now = 0

for fold in range(N_FOLDS):
    for seed in SEEDS_LIST:
        run_dir = training_dir / f'fold_{fold}_4sec' / f'seed={seed}'
        # The trainer writes a `done` file ONLY after the full nBatch loop
        # finishes. modelWeights alone is unreliable as a "complete" marker
        # because the trainer also saves it mid-training on accuracy improvements.
        if (run_dir / 'done').exists():
            print(f'[skip] fold={fold} seed={seed} (already done)')
            continue
        cfg = deepcopy(base_cfg)
        cfg['datasetPath'] = str(formatted_data_dir / f'all_fold_{fold}_4sec.pkl')
        cfg['outputDir']   = str(run_dir)
        cfg['seed']        = seed
        completed_now += 1
        print(f'[train {completed_now}/{n_total}] fold={fold} seed={seed} → {run_dir}')
        trainModel(cfg)

## Check completion

Verifies the expected number of runs produced `modelWeights` and `trainingStats`.

In [ ]:
import pickle

completed = []
missing   = []
for fold in range(N_FOLDS):
    for seed in SEEDS_LIST:
        run_dir = training_dir / f'fold_{fold}_4sec' / f'seed={seed}'
        if (run_dir / 'modelWeights').exists() and (run_dir / 'trainingStats').exists():
            with (run_dir / 'trainingStats').open('rb') as handle:
                stats = pickle.load(handle)
            completed.append((fold, seed, float(stats['testClassAcc'][-1])))
        else:
            missing.append((fold, seed, run_dir))

n_expected = N_FOLDS * len(SEEDS_LIST)
print(f'Completed runs: {len(completed)} / {n_expected}')
if completed:
    print(f'Held-out accuracy range: {min(x[2] for x in completed):.3f}-{max(x[2] for x in completed):.3f}')
if missing:
    print('Missing runs:')
    for fold, seed, run_dir in missing:
        print(f'  fold={fold}, seed={seed}: {run_dir}')